In [ ]:
# Install packages
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns

#Set file paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr/"data"/"raw"
processed_fldr = root_fldr/'data'/'processed'

#Load calflora data and convert coordinates to points
calflora1_raw = pd.read_csv(raw_fldr/"calflora-out.csv", encoding="cp1252")
calflora2_raw = pd.read_csv(raw_fldr/"calflora-out part 2.csv", encoding="cp1252")
calflora3_raw = pd.read_csv(raw_fldr/"calflora-out part 3.csv", encoding="cp1252")

calflora1 = gpd.GeoDataFrame(calflora1_raw, geometry=gpd.points_from_xy(calflora1_raw["longitude"], calflora1_raw["latitude"]),
    crs="EPSG:4326"
)
calflora2 = gpd.GeoDataFrame(calflora2_raw, geometry=gpd.points_from_xy(calflora2_raw["longitude"], calflora2_raw["latitude"]),
    crs="EPSG:4326"
)
calflora3 = gpd.GeoDataFrame(calflora3_raw, geometry=gpd.points_from_xy(calflora3_raw["longitude"], calflora3_raw["latitude"]),
    crs="EPSG:4326"
)

calflora_full = pd.concat([calflora1, calflora2, calflora3])

#Drop overlapped observation from final dataset
calflora_full = calflora_full.drop_duplicates()
calflora_full["date"] = pd.to_datetime(calflora_full['date'], format = '%m/%d/%Y')

#Load Calflora outline
cal_outline = gpd.read_file(processed_fldr / "cal_outline.gpkg")


In [3]:
calflora_full_cleaned = (
    calflora_full
    .drop(columns=["quality", "county", "number_of_plants"])
    .set_crs("EPSG:4326", allow_override=True)
            .to_crs("EPSG:3310")
    .assign(
            year=lambda df: df["date"].dt.year.astype(int),
            date=lambda df: df["date"].dt.tz_localize(None))
)

In [4]:
#Remove records outside of California
cal_outline = cal_outline.to_crs(calflora_full_cleaned.crs)
calflora_ca = gpd.sjoin(
    calflora_full_cleaned,
    cal_outline[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right")

In [6]:
calflora_full_cleaned.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 314827 entries, 0 to 99570
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id               314827 non-null  object        
 1   common_name      314438 non-null  object        
 2   scientific_name  314827 non-null  object        
 3   date             314827 non-null  datetime64[ns]
 4   latitude         314826 non-null  float64       
 5   longitude        314826 non-null  float64       
 6   geometry         314827 non-null  geometry      
 7   year             314827 non-null  int64         
dtypes: datetime64[ns](1), float64(2), geometry(1), int64(1), object(3)
memory usage: 21.6+ MB


In [7]:
#save to processed folder
calflora_full_cleaned.to_file(
    processed_fldr / "calflora_full_cleaned.gpkg",
    driver="GPKG"
)

In [9]:
calflora_full_cleaned["year"].value_counts()

year
2017    41347
2021    33367
2016    31113
2019    30009
2022    27959
2018    27111
2020    26305
2023    26044
2024    25305
2025    23816
2026    22451
Name: count, dtype: int64